# 📊 Master Pipeline Data Mining: Segmentasi UMKM Kota Bandung

Notebook ini merupakan **Master Execution Pipeline** yang mengintegrasikan alur pemrosesan data secara lengkap dari **Tahap 01 hingga Tahap 05**.

---
### 🔄 Tahapan Pipeline yang Dieksekusi:
1. **Tahap 01 (Preprocessing Member)**: Preprocessing data scraper mentah 3 anggota (Indra, Dwi, Rajif).
2. **Tahap 02 (Merger & Deduplikasi)**: Penggabungan data & deduplikasi entitas UMKM Kota Bandung (`02_Data_Final_Sebelum_NLP_V2.csv`).
3. **Tahap 03 (NLP Sentiment)**: Analisis sentimen ulasan UMKM (`03_Data_Modeling_Setelah_NLP.csv`).
4. **Tahap 04 (K-Means Clustering & PCA)**: Pemodelan K-Means ($K=3$), proyeksi PCA 2D, dan evaluasi Silhouette Score (`04_Hasil_Clustering_Final.csv`).
5. **Tahap 05 (Rekomendasi Bisnis Strategis)**: Penjanaan rekomendasi bisnis berbasis segmen kluster (`05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.csv`).


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Hubungkan modul dari src/
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("../src"))

from data_loader import load_member_raw_data, load_csv_safe
from preprocessing import clean_member_data, merge_and_deduplicate
from nlp_analysis import run_nlp_sentiment_analysis
from model import run_kmeans_clustering, generate_llm_recommendations
from utils import save_stage_csv, print_stage_header

print("[OK] Pustaka dan modul berhasil dimuat.")


### 📌 Tahap 01: Preprocessing Data Scraper Mentah (Indra, Dwi, Rajif)


In [ ]:
print_stage_header(1, "Preprocessing Data Scraper Mentah 3 Anggota")

df_info_indra, df_rev_indra = load_member_raw_data("indra")
df_clean_indra = clean_member_data(df_info_indra, df_rev_indra, "Indra")
out_01_indra = save_stage_csv(df_clean_indra, "01_Hasil_Preprocessing_Indra_Final.csv")
print(f"  [OK] Indra: {len(df_clean_indra):,} baris -> {out_01_indra}")

df_info_dwi, df_rev_dwi = load_member_raw_data("dwi")
df_clean_dwi = clean_member_data(df_info_dwi, df_rev_dwi, "Dwi")
out_01_dwi = save_stage_csv(df_clean_dwi, "01_Hasil_Preprocessing_Dwi_Final.csv")
print(f"  [OK] Dwi: {len(df_clean_dwi):,} baris -> {out_01_dwi}")

df_info_rajif, df_rev_rajif = load_member_raw_data("rajif")
df_clean_rajif = clean_member_data(df_info_rajif, df_rev_rajif, "Rajif")
out_01_rajif = save_stage_csv(df_clean_rajif, "01_Hasil_Preprocessing_Rajif_Final.csv")
print(f"  [OK] Rajif: {len(df_clean_rajif):,} baris -> {out_01_rajif}")


### 📌 Tahap 02: Penggabungan & Deduplikasi Lintas Anggota


In [ ]:
print_stage_header(2, "Penggabungan & Deduplikasi Lintas Anggota")

df_merged = merge_and_deduplicate(df_clean_indra, df_clean_dwi, df_clean_rajif)
if df_merged.empty:
    df_merged = load_csv_safe("data_umkm_bandung.csv")

out_02 = save_stage_csv(df_merged, "02_Data_Final_Sebelum_NLP_V2.csv")
print(f"  [OK] Merger & Deduplikasi: {len(df_merged):,} entitas unik -> {out_02}")
display(df_merged.head())


### 📌 Tahap 03: Pemrosesan NLP Sentimen Ulasan UMKM


In [ ]:
print_stage_header(3, "Pemrosesan NLP Sentimen Ulasan UMKM")

df_nlp = run_nlp_sentiment_analysis(df_merged)
out_03 = save_stage_csv(df_nlp, "03_Data_Modeling_Setelah_NLP.csv")
print(f"  [OK] Analisis Sentimen NLP: Rerata Skor = {df_nlp['sentiment_score'].mean():.4f} -> {out_03}")
display(df_nlp[['title', 'sentiment_score', 'text']].head())


### 📌 Tahap 04: Pemodelan K-Means Clustering & PCA 2D


In [ ]:
print_stage_header(4, "Pemodelan K-Means Clustering & PCA 2D")

df_cluster, kmeans_model, metrics, features, X_scaled = run_kmeans_clustering(df_nlp)
out_04 = save_stage_csv(df_cluster, "04_Hasil_Clustering_Final.csv")

print(f"  [OK] Clustering K-Means (K=3) Selesai -> {out_04}")
print(f"       • Silhouette Score     : {metrics['silhouette_score']:.4f}")
print(f"       • Davies-Bouldin Index : {metrics['davies_bouldin_score']:.4f}")
print(f"       • Calinski-Harabasz    : {metrics['calinski_harabasz_score']:.4f}")

# Visualisasi Scatter Plot PCA 2D
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_cluster, x='pca_x', y='pca_y', hue='cluster_name',
    palette='Set2', style='cluster_name', s=80, alpha=0.85
)
plt.title('Segmentasi UMKM Kota Bandung (Proyeksi PCA 2D)', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### 📌 Tahap 05: Penjanaan Rekomendasi Bisnis Strategis (LLM)


In [ ]:
print_stage_header(5, "Penjanaan Rekomendasi Bisnis Strategis (LLM)")

df_final = generate_llm_recommendations(df_cluster)
out_05 = save_stage_csv(df_final, "05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.csv")
out_segmented = save_stage_csv(df_final, "data_umkm_segmented.csv")

print(f"  [OK] Rekomendasi Bisnis Selesai -> {out_05}")
print(f"  [OK] Dataset Tersegmentasi -> {out_segmented}")

print("
🎉 [SELESAI] Master Pipeline 5-Tahap Berhasil Dieksekusi 100%!")
display(df_final[['title', 'cluster_name', 'rekomendasi_strategis']].head(10))
